# How to define custom neural nets

`sbi` allows you to specify a specific density estimator for each of the implemented methods.
We support a variety of density estimators, e.g., mixtures of Gaussians, normalizing
flows, and diffusion models. Some of the density estimators are implemented as part of
`sbi`, for others we rely on other packages like
[`nflows`](https://github.com/bayesiains/nflows/) or [`zuko`](https://github.com/probabilists/zuko). 

For all options, check the API reference
[here](https://sbi.readthedocs.io/en/latest/api_reference.html).


## Changing the type of density estimator

One option is using one of the preconfigured density estimators by passing a string in
the `density_estimator` keyword argument to the inference object (`NPE` or `NLE`), e.g.,
"maf" for a Masked Autoregressive Flow, of "nsf" for a Neural Spline Flow with default
hyperparameters.

**New with sbi 0.23:** Note that `"maf"` or `"nsf"` correspond to `nflows` density
estimators. Those have proven to work well, but the `nflows` package is not maintained
anymore. To use more recent and actively maintained density estimators, we tentatively
recommend using `zuko`, e.g., by passing `zuko_maf` or `zuko_nsf`. 


In [1]:
import torch

from sbi.inference import NPE, NRE
from sbi.utils import BoxUniform

In [2]:
prior = BoxUniform(torch.zeros(2), torch.ones(2))
inference = NPE(prior=prior, density_estimator="zuko_maf")

In the case of `NRE`, the argument is called `classifier`:


In [3]:
inference = NRE(prior=prior, classifier="resnet")

## Changing hyperparameters of density estimators


Alternatively, you can configure a density estimator yourself with a per-model config from `sbi.neural_nets`, e.g. to use a flow with hyperparameters chosen for your problem at hand.

Here, because we want to use N*P*E, we pick the config modelling the _posterior_ density $p(\theta|x)$. In this example, we will create a Zuko neural spline flow (`ZukoNSFConfig`) with `60` hidden units and `3` transform layers:

In [4]:
# For SNLE: a density config, e.g. ZukoNSFConfig(). For SNRE: a classifier config.
from sbi.neural_nets import ZukoNSFConfig

density_estimator = ZukoNSFConfig(hidden_features=60, num_transforms=3)
inference = NPE(prior=prior, density_estimator=density_estimator)

It is also possible to pass an `embedding_net` to the config to automatically
learn summary statistics from high-dimensional simulation outputs. You can find a more
detailed tutorial on this in [04_embedding_networks](https://sbi.readthedocs.io/en/latest/how_to_guide/04_embedding_networks.html).

> **Deprecated:** the `posterior_nn`, `likelihood_nn`, `classifier_nn`, `posterior_score_nn`, `posterior_flow_nn`, and `marginal_nn` factory functions are deprecated since sbi v0.28.0 and will be removed in v0.29.0. Pass the config itself instead, e.g. `ZukoNSFConfig(...)` where you would have passed `posterior_nn(model="zuko_nsf", ...)`. The behaviour is identical.

## Building new density estimators from scratch


Finally, it is also possible to implement your own density estimator from scratch, e.g., including embedding nets to preprocess data, or to a density estimator architecture of your choice.

For this, the `density_estimator` argument needs to be a function that takes `theta` and `x` batches as arguments to then construct the density estimator after the first set of simulations was generated. The per-model configs in `sbi.neural_nets` return such a function from their `build` method, so `SomeConfig(...).build` is a valid `density_estimator`.

The returned `density_estimator` object needs to be a subclass of `DensityEstimator`, which requires to implement three methods:
    
- `log_prob(input, condition, **kwargs)`: Return the log probabilities of the inputs given a condition or multiple i.e. batched conditions.
- `loss(input, condition, **kwargs)`: Return the loss for training the density estimator.
- `sample(sample_shape, condition, **kwargs)`: Return samples from the density estimator.

See more information on the [Reference API page](https://sbi.readthedocs.io/en/latest/api_reference.html).